<a href="https://colab.research.google.com/github/carlduya-tech/Prompt-Engineering/blob/main/Carl_Duya_Prompt_Engineering_Exercise_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Mock LLM: AI-in-Schools Summary Bot (user-friendly + self-critique + refinement)

What you get:
- A small local "mock LLM" (rule-based + templated) that can:
  - Return 3 bullet points per topic (3 topics)
  - Return a verification section (objective vs interpretive)
  - Provide a combined summary
- A friendly CLI with:
  - Suggested inputs
  - Commands (/help, /examples, /mode, /quit)
- Self-assessment loop:
  - Draft answer
  - Critique (checks clarity, redundancy, completeness, constraints)
  - Refines once (and only once) when it can improve

No web browsing. No external dependencies.
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import List, Tuple, Optional
import textwrap


# -----------------------------
# Knowledge Base (the "model")
# -----------------------------

@dataclass(frozen=True)
class Topic:
    name: str
    bullets: Tuple[str, str, str]


@dataclass(frozen=True)
class Verification:
    verifiable_attributed: Tuple[str, ...]
    interpretive_normative: Tuple[str, ...]
    caveats: Tuple[str, ...]


TOPICS: Tuple[Topic, ...] = (
    Topic(
        name="Student learning support (reading & writing)",
        bullets=(
            "Reading/leveling text: AI can adjust passage difficulty/complexity to match a student's reading level.",
            "Language acquisition: teachers report AI can help second-language learners with tailored practice and input.",
            "Writing support: used as a coach for brainstorming, drafting (organization/coherence/grammar), and revision/editing.",
        ),
    ),
    Topic(
        name="Teacher productivity and classroom operations",
        bullets=(
            "Parent communication: drafting messages/emails to families.",
            "Materials creation: generating worksheets, rubrics, quizzes, and lesson-plan drafts.",
            "Translation/adaptation: translating or adapting materials for multilingual classrooms and families.",
        ),
    ),
    Topic(
        name="Access, inclusion, and equity-oriented deployments",
        bullets=(
            "Reaching excluded learners: cited example of distributing curriculum content/lessons via WhatsApp in multiple languages.",
            "Accessibility/learning differences: potential support for disabilities and learning differences (e.g., dyslexia).",
            "Equity risk: free tools may be less reliable while higher-quality models can cost more—widening resource gaps.",
        ),
    ),
)

VERIFICATION = Verification(
    verifiable_attributed=(
        "Scope claims (e.g., number of countries, volume of literature) are factual *as attributed* to the report/article; confirm by checking the primary report.",
        "Teacher time-savings claims (e.g., ~6 hours/week) are factual *as attributed* to a referenced study/poll; confirm by reviewing the original methods.",
        "Survey stats about youth AI companionship/romantic framing are factual *as attributed* to a referenced survey; confirm via the survey instrument and sampling.",
    ),
    interpretive_normative=(
        "Claims like “risks outweigh benefits” or “grave threat” are normative conclusions, not objective measurements.",
        "Claims of broad learning declines are evidence syntheses/interpretations; objective validation requires evaluating underlying studies (causality, effect size, replication).",
    ),
    caveats=(
        "A news article is a secondary source; corroborate quantitative claims with primary sources.",
        "Even correct numbers can be misread without context (population, measures, timeframe, correlation vs causation).",
    ),
)


# -----------------------------
# Formatting helpers
# -----------------------------

def _wrap(text: str, width: int = 92) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


def format_topics() -> str:
    out: List[str] = []
    for t in TOPICS:
        out.append(f"## {t.name}")
        out.extend([f"- {b}" for b in t.bullets])
        out.append("")
    return "\n".join(out).strip()


def format_verification() -> str:
    v = VERIFICATION
    out: List[str] = []
    out.append("## Verification of data points (truth/objectivity)\n")
    out.append("### Verifiable factual claims (as attributed)")
    out.extend([f"- {x}" for x in v.verifiable_attributed])
    out.append("")
    out.append("### Interpretive / normative claims (not purely objective)")
    out.extend([f"- {x}" for x in v.interpretive_normative])
    out.append("")
    out.append("### Caveats")
    out.extend([f"- {x}" for x in v.caveats])
    return "\n".join(out).strip()


def format_summary() -> str:
    return "\n\n".join([format_topics(), format_verification()])


# -----------------------------
# Intent + Modes
# -----------------------------

class Mode(str, Enum):
    NORMAL = "normal"       # answer only
    VERBOSE = "verbose"     # answer + brief "how to use" reminder
    QUIET = "quiet"         # minimal framing


EXAMPLES = (
    "topics",
    "verification",
    "summary",
    "What are the 3 topics of AI use in schools?",
    "Verify which claims are objective vs interpretive.",
)

HELP_TEXT = _wrap(
    """\
Commands:
  /help        Show this help
  /examples    Show example inputs
  /mode        Show current mode
  /mode quiet  Minimal responses
  /mode normal Default responses
  /mode verbose Add a small usage hint after each response
  /quit        Exit

Natural language is fine too. Suggested inputs:
  - "topics"        -> 3 bullets per topic about AI use in schools
  - "verification"  -> objective vs interpretive claims
  - "summary"       -> both together
"""
)


def classify_intent(user_text: str) -> str:
    u = user_text.strip().lower()
    if u in ("topics", "uses", "use cases"):
        return "topics"
    if u in ("verification", "verify", "objectivity"):
        return "verification"
    if u in ("summary", "summarize", "bullets"):
        return "summary"

    # lightweight keyword routing
    if any(k in u for k in ("verify", "objective", "true", "data point", "factual", "interpret")):
        return "verification"
    if any(k in u for k in ("topic", "topics", "use cases", "being used", "uses in school")):
        return "topics"
    if any(k in u for k in ("summary", "summarize", "bullet", "key points")):
        return "summary"

    return "help"


# -----------------------------
# Self-critique + refinement
# -----------------------------

@dataclass
class Critique:
    issues: List[str]
    improved_answer: Optional[str] = None


def critique_and_refine(intent: str, draft: str) -> Critique:
    """
    A tiny "reflection" pass:
    - checks constraints (3 bullets per topic)
    - checks redundancy (duplicate lines)
    - checks clarity (overly long lines)
    If it can improve, returns a refined answer.
    """

    issues: List[str] = []

    # 1) Constraint: 3 bullets per topic when intent includes topics/summary
    if intent in ("topics", "summary"):
        # Count bullets under each topic heading
        blocks = [b for b in draft.split("\n## ") if b.strip()]
        # The first block might start with "## " already; normalize:
        if draft.startswith("## "):
            blocks = [draft[3:]]  # remove leading "## "
            blocks = blocks[0].split("\n## ")
        for blk in blocks:
            # Only evaluate topic sections (ignore verification sections)
            header = blk.splitlines()[0].strip()
            if header.lower().startswith("verification"):
                continue
            bullet_count = sum(1 for line in blk.splitlines() if line.strip().startswith("- "))
            if bullet_count and bullet_count != 3:
                issues.append(f"Topic '{header}' has {bullet_count} bullets; expected 3.")

    # 2) Redundancy
    lines = [ln.strip() for ln in draft.splitlines() if ln.strip()]
    if len(lines) != len(set(lines)):
        issues.append("Some lines are duplicated; response can be tightened.")

    # 3) Clarity: too-long bullets
    long_bullets = [ln for ln in draft.splitlines() if ln.strip().startswith("- ") and len(ln) > 140]
    if long_bullets:
        issues.append("Some bullets are very long; could be split or shortened for readability.")

    if not issues:
        return Critique(issues=[])

    # If improvements are possible, do ONE refinement pass.
    improved = draft

    # Remove duplicate lines while preserving order (safe tightening)
    seen = set()
    deduped: List[str] = []
    for ln in improved.splitlines():
        key = ln.strip()
        if not key:
            deduped.append(ln)
            continue
        if key in seen:
            continue
        seen.add(key)
        deduped.append(ln)
    improved = "\n".join(deduped).strip()

    # Shorten long bullets slightly (basic heuristics)
    # (We avoid altering meaning; just trim parentheticals and tighten phrasing.)
    improved_lines: List[str] = []
    for ln in improved.splitlines():
        s = ln.strip()
        if s.startswith("- ") and len(s) > 140:
            s = s.replace("used as a coach for", "used to")
            s = s.replace("generating or adapting", "translating/adapting")
            s = s.replace("—", "-")
            # keep it reasonable
            if len(s) > 140:
                s = s[:137].rstrip() + "..."
            improved_lines.append(s)
        else:
            improved_lines.append(ln)
    improved = "\n".join(improved_lines).strip()

    return Critique(issues=issues, improved_answer=improved)


# -----------------------------
# "Mock LLM" generation
# -----------------------------

def generate(intent: str) -> str:
    if intent == "topics":
        return format_topics()
    if intent == "verification":
        return format_verification()
    if intent == "summary":
        return format_summary()
    return HELP_TEXT


def respond(user_text: str, mode: Mode) -> str:
    intent = classify_intent(user_text)
    draft = generate(intent)

    critique = critique_and_refine(intent, draft)
    final = critique.improved_answer if critique.improved_answer else draft

    # User-friendly postscript
    if mode == Mode.VERBOSE and intent != "help":
        final += "\n\n" + _wrap("Tip: try 'topics', 'verification', or 'summary' — or type /examples.")

    # In QUIET mode, strip help headers if possible (but keep content)
    if mode == Mode.QUIET and intent == "help":
        final = "Try: topics | verification | summary  (or /examples, /help, /quit)"

    # Optional: show the self-assessment (lightweight, not verbose)
    # If you want it visible, flip SHOW_SELF_ASSESSMENT to True.
    SHOW_SELF_ASSESSMENT = False
    if SHOW_SELF_ASSESSMENT and critique.issues:
        assessment = "\n".join([f"- {i}" for i in critique.issues])
        final += "\n\n" + "Self-assessment:\n" + assessment

    return final


# -----------------------------
# CLI chat loop
# -----------------------------

BANNER = _wrap(
    """\
Mock LLM ready.

Suggested inputs:
  - topics
  - verification
  - summary

Commands:
  /help, /examples, /mode [quiet|normal|verbose], /quit
"""
)


def chat() -> None:
    mode = Mode.NORMAL
    print(BANNER)

    while True:
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            return

        if not user_text:
            print("Assistant: (type /examples for ideas)")
            continue

        # Commands
        if user_text.lower() in ("/quit", "quit", "exit"):
            print("Assistant: Bye.")
            return

        if user_text.lower() == "/help":
            print("Assistant:\n" + HELP_TEXT + "\n")
            continue

        if user_text.lower() == "/examples":
            ex = "\n".join(f"- {e}" for e in EXAMPLES)
            print("Assistant:\n" + _wrap("Example inputs:\n" + ex) + "\n")
            continue

        if user_text.lower().startswith("/mode"):
            parts = user_text.split()
            if len(parts) == 1:
                print(f"Assistant: Current mode = {mode.value}\n")
                continue
            desired = parts[1].lower()
            if desired in (m.value for m in Mode):
                mode = Mode(desired)  # type: ignore[arg-type]
                print(f"Assistant: Mode set to {mode.value}\n")
            else:
                print("Assistant: Unknown mode. Use /mode quiet | normal | verbose\n")
            continue

        # Normal response
        answer = respond(user_text, mode)
        print("Assistant:\n" + answer + "\n")


if __name__ == "__main__":
    chat()

Mock LLM ready.

Suggested inputs:
  - topics
  - verification
  - summary

Commands:
  /help, /examples, /mode [quiet|normal|verbose], /quit
You: topics
Assistant:
## Student learning support (reading & writing)
- Reading/leveling text: AI can adjust passage difficulty/complexity to match a student's reading level.
- Language acquisition: teachers report AI can help second-language learners with tailored practice and input.
- Writing support: used as a coach for brainstorming, drafting (organization/coherence/grammar), and revision/editing.

## Teacher productivity and classroom operations
- Parent communication: drafting messages/emails to families.
- Materials creation: generating worksheets, rubrics, quizzes, and lesson-plan drafts.
- Translation/adaptation: translating or adapting materials for multilingual classrooms and families.

## Access, inclusion, and equity-oriented deployments
- Reaching excluded learners: cited example of distributing curriculum content/lessons via Whats